# Van3Twin Simulator — Ookayama Scenario

## 1 · Scene & Environment Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import numpy as np
import pandas as pd
import json
import csv as _csv
import sim_startup as sim
import core.rt as rt
import network.nr as nr
from sionna.rt import load_mesh, SceneObject, ITURadioMaterial

path_suffix = "/home/rpegurri"

# ── Ray-tracing scene ─────────────────────────────────────────────────────────
scene = sim.setup_scene(
    file_name=f"{path_suffix}/tokyo-polimi-dt-jsac/scenarios/ookayama_full_flat/ookayama.xml",
    frequency=15e9,
    bandwidth=40e6,
)

sim.setup_antennas(
    transmitters=np.arange(100),   # vehicles: IDs 0–99
    receivers=np.arange(120),      # vehicles + RSUs: IDs 0–119
    pattern="dipole",
    polarization="VH",
)

sim.configure_rt(
    verbose=False,
    time_checker=False,
    rt_diffraction=False,
    rt_corner_diffraction=False,
    rt_max_depth=4,
)

struc = sim.startup()

## 2 · Helper Functions

In [ ]:
# ── Asset paths & TX powers ───────────────────────────────────────────────────
COMS_MESH        = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/coms.ply"
ESTIMA_MESH      = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/estima.ply"
RSU_MESH         = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/rsu.ply"
CAR_TX_POWER_DBM = 23
RSU_TX_POWER_DBM = 30


def add_new_car(i):
    if scene.get(f"obj_{i}") is not None:
        print(f"Car {i} (obj_{i}) already exists in the scene.")
        return scene.get(f"obj_{i}")
    
    # coms if i is odd, Estima if i is even
    if i % 2 == 0:
        car_mesh = load_mesh(COMS_MESH)
        sim.set_antenna_displacement(i, [0.096, 0.157001, 0.791505]) # TOYOTA COMS antennas

    else:
        car_mesh = load_mesh(ESTIMA_MESH)
        sim.set_antenna_displacement(i, [-0.281679, 0.477371, 0.975445]) # TOYOTA ESTIMA antennas

    car_obj = SceneObject(mi_mesh=car_mesh,
                          name=f"obj_{i}",
                          radio_material=ITURadioMaterial(f"itu_metal_{i}",
                                                        "metal",
                                                        thickness=0.01,
                                                        color=(0.8, 0.1, 0.1)))
    scene.edit(add=car_obj)

    # Set the car transmit power
    sim.set_tx_power(i, CAR_TX_POWER_DBM)

    return car_obj

def add_roads():
    roads_mesh = load_mesh(f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/roads.ply")
    roads_obj = SceneObject(mi_mesh=roads_mesh,
                          name="roads",
                          radio_material=ITURadioMaterial("roads_material", "concrete", thickness=0.1, color=(0.6, 0.6, 0.6)))
    scene.edit(add=roads_obj)

def add_rsu():
    rsu = {}
    z_rsu = 5 # height of the RSU antennas

    # OSM Scenario
    #rsu[1] = [117.003, 182.285, 0.000]
    #rsu[2] = [-18.178, 153.100, 0.000]
    #rsu[3] = [72.637, 137.367, 0.000]
    #rsu[4] = [122.737, 123.874, 0.000]
    #rsu[5] = [74.775, 90.218, 0.000]
    #rsu[6] = [45.752, 30.378, 0.000]
    #rsu[7] = [-37.893, 24.764, 0.000]
    #rsu[8] = [92.606, 7.428, 0.000]
    #rsu[9] = [-49.461, -67.075, 0.000]
    #rsu[10] = [76.155, -93.238, 0.000]
    #rsu[11] = [21.455, -105.447, 0.000]
    #rsu[12] = [-53.560, -135.201, 0.000]
    #rsu[13] = [25.993, -145.066, 0.000]
    #rsu[14] = [71.552, -144.068, 0.000]
    #rsu[15] = [-14.368, -157.715, 0.000]
    #rsu[16] = [-37.711, -189.516, 0.000]
    #rsu[17] = [16.430, -206.697, 0.000]
    #rsu[18] = [63.666, -213.443, 0.000]
    #rsu[19] = [-24.461, -218.899, 0.000]

    # Full HQ Scenario
    rsu[1] = [235.030, 241.847, 0.000]
    rsu[2] = [104.836, 227.818, 0.000]
    rsu[3] = [196.304, 197.874, 0.000]
    rsu[4] = [244.342, 177.640, 0.000]
    rsu[5] = [194.801, 145.882, 0.000]
    rsu[6] = [166.806, 91.554, 0.000]
    rsu[7] = [84.763, 87.369, 0.000]
    rsu[8] = [214.241, 69.457, 0.000]
    rsu[9] = [73.434, -5.465, 0.000]
    rsu[10] = [198.417, -33.446, 0.000]
    rsu[11] = [139.264, -42.384, 0.000]
    rsu[12] = [69.375, -69.949, 0.000]
    rsu[13] = [147.666, -83.951, 0.000]
    rsu[14] = [190.319, -114.190, 0.000]
    rsu[15] = [108.298, -95.250, 0.000]
    rsu[16] = [84.493, -126.895, -0.000]
    rsu[17] = [140.243, -149.324, 0.000]
    rsu[18] = [186.151, -150.019, 0.000]
    rsu[19] = [95.546, -161.824, 1.000]

    for i in range(1, 20):
        x = rsu[i][0]
        y = rsu[i][1]

        rsu_mesh = load_mesh(RSU_MESH)
        rsu_obj = SceneObject(mi_mesh=rsu_mesh,
                          name=f"obj_{i+100}",
                          radio_material=ITURadioMaterial(f"itu_metal_{i+100}",
                                                        "metal",
                                                        thickness=0.01,
                                                        color=(0.8, 0.1, 0.1)))
        scene.edit(add=rsu_obj)

        # Set the RSU location
        loc_update_message = f"LOC_UPDATE:obj{100+i},{x},{y},{z_rsu/2},0,0,0,0"
        sim.set_antenna_displacement(100+i, [0, 0, z_rsu/2 + 0.5]) # RSU antennas
        rt.manage_location_message(loc_update_message, struc)
        #print(f"    [INFO] Added RSU {i} (obj_{100+i}) at location ({x}, {y}, {z_rsu}).")

        # Set the RSU transmit power
        sim.set_tx_power(100+i, RSU_TX_POWER_DBM)
    return rsu

#add_roads()
rsu = add_rsu()

## 3 · Simulation Parameters

In [ ]:
# ── NR sidelink Mode 2 (TS 38.321 §5.22.1.1) ─────────────────────────────────
# Nodes the TX can sense above this threshold are excluded from the candidate
# interferer set via SPS sensing; only hidden nodes (below this) can collide.
SL_THRESH_S_RSRP_DBM = -110.0   # dBm

# ── Network occupancy (Channel Busy Ratio, TS 38.321 §5.22.1.3) ───────────────
# ρ ∈ [0, 1]: probability that a given node is actively occupying the resource
# pool at any timestep.
#   None  → dynamic: min(1.0, n_active_cars / N_RSUS)
#           RSU activity scales with vehicle density — with few cars, RSUs are
#           unlikely to be relaying; they become active as the network fills up.
#   float → fixed rate for every timestep (e.g. 1.0 = all nodes always active).
OCCUPANCY_RATE = 0.33

# ── Output ────────────────────────────────────────────────────────────────────
_CSV_PATH   = "simulation_dataset.csv"
_CSV_FIELDS = [
    "timestamp", "tx_id", "rx_id",
    "is_los", "rssi_dbm", "sinr_eff_db", "mcs_index", "modulation",
    "bler", "throughput_kbps", "tx_x", "tx_y", "rx_x", "rx_y",
]

## 4 · Run Simulation and start visualizer

Visualizer runs at: http://localhost:8000

In [ ]:
import visualizer.manager as vis
vis.start_server()

df              = pd.read_csv(f"{path_suffix}/tokyo-polimi-dt-jsac/ookayama_sumo/traces.csv", sep=";")
active_vehicles = set(range(101, 120))   # RSU IDs 101-119 are always present
output          = {}

starting_timestamp = 522.8

# SUMO coordinate offset: align SUMO origin to Sionna scene origin (spoiler: is wrong)
#x_off = 113.708 - 367.05
#y_off = 109.745 - 741.32

# Offset for full HQ scenario
x_off = 188.524 - 317.26
y_off = -163.742 - 406.5

N_RSUS        = len(rsu)
_activity_rng = np.random.default_rng(42)   # seeded once; drives per-timestep node activity

# Open streaming CSV - rows flushed immediately so file is readable mid-run
_csv_fh     = open(_CSV_PATH, "w", newline="")
_csv_writer = _csv.DictWriter(_csv_fh, fieldnames=_CSV_FIELDS)
_csv_writer.writeheader()
_csv_fh.flush()

# -- Main loop -----------------------------------------------------------------
for timestamp, group in df.groupby("timestep_time"):

    if timestamp < starting_timestamp:
        #print(f"Skipping timestamp {timestamp} to speed up the simulation.")
        continue

    output[timestamp] = {"mobility": {}, "kpi": {}}
    evaluated_pairs  = set()
    vehicle_positions = {}   # {v_id: {"x", "y", "heading", "speed"}}

    print(f"Processing timestamp: {timestamp} ({len(group)} vehicles)")

    # -- Mobility update --------------------------------------------------------
    for _, row in group.iterrows():
        v_id  = int(row["vehicle_id"])
        x, y  = row["vehicle_x"], row["vehicle_y"]
        angle = row["vehicle_angle"]
        speed = row["vehicle_speed"]
        z     = 0.758495 if v_id % 2 == 0 else 0.874555   # COMS / Estima height

        heading = -angle * np.pi / 180
        v_x, v_y = speed * np.cos(heading), speed * np.sin(heading)

        vehicle_positions[v_id] = {"x": x + x_off, "y": y + y_off,
                                   "heading": heading, "speed": speed}

        loc_msg = f"LOC_UPDATE:obj{v_id},{x + x_off},{y + y_off},{z},{angle},{v_x},{v_y},0"
        if v_id not in active_vehicles:
            print(f"  + New vehicle: {v_id}")
            add_new_car(v_id)
            active_vehicles.add(v_id)
        rt.manage_location_message(loc_msg, struc)

    # -- Log mobility -----------------------------------------------------------
    for v_id, pos in vehicle_positions.items():
        output[timestamp]["mobility"].update({
            f"car_{v_id}_position_x": pos["x"],
            f"car_{v_id}_position_y": pos["y"],
            f"car_{v_id}_heading":    pos["heading"],
            f"car_{v_id}_speed":      pos["speed"],
        })

    # -- Node activity (CBR model) ---------------------------------------------
    # Each node draws activity once per timestep - same result used for every link
    # so that uplink and downlink see the identical set of active interferers.
    n_active_cars = len([v for v in active_vehicles if v < 100])
    occupancy_rate = (OCCUPANCY_RATE if OCCUPANCY_RATE is not None
                      else min(1.0, n_active_cars / N_RSUS))
    node_is_active = {v: bool(_activity_rng.random() < occupancy_rate)
                      for v in active_vehicles}

    # -- Path-loss matrix -------------------------------------------------------
    # Stores {(tx, rx): {"rssi": float, "is_los": bool}} for all reachable pairs.
    # RSU-RSU links are excluded (no RSU<->RSU sidelink communication).
    # Path loss is symmetric (TDD); RSSI differs per direction (TX powers differ).
    rssi_matrix = {}

    for car_id in active_vehicles:
        for other_car_id in active_vehicles:
            if car_id == other_car_id:
                continue
            if (car_id, other_car_id) in evaluated_pairs or (other_car_id, car_id) in evaluated_pairs:
                continue
            if car_id >= 100 and other_car_id >= 100:
                continue   # no RSU-RSU links

            pl = float(rt.get_path_loss(car_id, other_car_id, struc))
            evaluated_pairs.add((car_id, other_car_id))

            if pl != 404:   # 404 -> unreachable
                is_los = rt.get_los(car_id, other_car_id, struc)
                rssi_matrix[(car_id, other_car_id)] = {
                    "rssi": struc["tx_powers"][car_id] - pl, "is_los": is_los,
                }
                rssi_matrix[(other_car_id, car_id)] = {
                    "rssi": struc["tx_powers"][other_car_id] - pl, "is_los": is_los,
                }

    # -- KPI computation --------------------------------------------------------
    for (car_id, other_car_id), link in rssi_matrix.items():
        rssi   = link["rssi"]
        is_los = link["is_los"]

        # NR sidelink Mode 2 Sensing-Based SPS (TS 38.321 section 5.22.1.1):
        # hidden nodes are those the TX cannot sense above SL_THRESH_S_RSRP_DBM
        # and that are currently active in the resource pool.
        interferer_rssis = [
            v["rssi"]
            for (tx, rx), v in rssi_matrix.items()
            if rx == other_car_id
            and tx != car_id
            and node_is_active.get(tx, False)
            and rssi_matrix.get((tx, car_id), {"rssi": -999.0})["rssi"] < SL_THRESH_S_RSRP_DBM
        ]

        sinr_per_rb, sinr_eff, n_rbs = nr.compute_sinr(rssi_dbm=rssi,
                                                   bandwidth_hz=struc["bandwidth"],
                                                   interferer_rssis=interferer_rssis)

        result = nr.compute_nr_mcs(sinr_db=sinr_eff,
                                   target_bler=0.1,
                                   cbs=5504)

        thput = nr.compute_nr_thput(spectral_efficiency=result['spectral_eff'],
                                    bandwidth_hz=struc["bandwidth"],
                                    num_streams=2)

        type_tx  = "car" if car_id < 100 else "rsu"
        type_rx  = "car" if other_car_id < 100 else "rsu"
        label_tx = car_id if car_id < 100 else car_id - 100
        label_rx = other_car_id if other_car_id < 100 else other_car_id - 100

        output[timestamp]["kpi"][f"{type_tx}_{label_tx}_to_{type_rx}_{label_rx}"] = {
            "rssi": rssi, "sinr": sinr_eff, "mcs_index": result['mcs'],
            "bler": result['predicted_bler'], "thput": thput, "modulation": result['modulation'],
            "spectral_efficiency": result['spectral_eff'], "is_los": is_los,
        }

        pos_tx = ({"x": rsu[label_tx][0], "y": rsu[label_tx][1]} if type_tx == "rsu"
                  else vehicle_positions.get(car_id, {}))
        pos_rx = ({"x": rsu[label_rx][0], "y": rsu[label_rx][1]} if type_rx == "rsu"
                  else vehicle_positions.get(other_car_id, {}))

        _csv_writer.writerow({
            "timestamp":      timestamp,
            "tx_id":          f"{type_tx}_{label_tx}",
            "rx_id":          f"{type_rx}_{label_rx}",
            "is_los":         int(is_los),
            "rssi_dbm":       round(rssi, 2),
            "sinr_eff_db":    round(sinr_eff, 2),
            "mcs_index":      result['mcs'],
            "modulation":     result['modulation'],
            "bler":           round(result['predicted_bler'], 6),
            "throughput_kbps": round(thput / 1e3, 2),
            "tx_x":           round(pos_tx.get("x", float("nan")), 3),
            "tx_y":           round(pos_tx.get("y", float("nan")), 3),
            "rx_x":           round(pos_rx.get("x", float("nan")), 3),
            "rx_y":           round(pos_rx.get("y", float("nan")), 3),
        })
        _csv_fh.flush()

    # Persist intermediate JSON so the file is readable if the run is interrupted
    with open("simulation_output.json", "w") as f:
        json.dump(output, f, indent=4)

    print(f"  Done. {len(rssi_matrix)//2} links evaluated.")

_csv_fh.close()
print(f"Simulation complete. Dataset saved to {_CSV_PATH}.")

## 5 · Scene Preview

In [ ]:
scene.preview(show_orientations=False, resolution=(1440, 1100))

## 6 · Movie mode
Rotto - Non da output, lavora in background

In [ ]:
df              = pd.read_csv(f"{path_suffix}/Tokyo JSAC/ookayama_sumo/traces.csv", sep=";")
active_vehicles = set(range(101, 120))   # RSU IDs 101–119 are always present
output          = {}

# Offset for full HQ scenario
x_off = 188.524 - 317.26
y_off = -163.742 - 406.5

# ── Main loop ─────────────────────────────────────────────────────────────────
for timestamp, group in df.groupby("timestep_time"):

    vehicle_positions = {}   # {v_id: {"x", "y", "heading", "speed"}}

    print(f"Processing frame export for timestamp: {timestamp} ({len(group)} vehicles)")

    for _, row in group.iterrows():
        v_id  = int(row["vehicle_id"])
        x, y  = row["vehicle_x"], row["vehicle_y"]
        angle = row["vehicle_angle"]
        speed = row["vehicle_speed"]
        z     = 0.758495 if v_id % 2 == 0 else 0.874555   # COMS / Estima height

        heading = -angle * np.pi / 180
        v_x, v_y = speed * np.cos(heading), speed * np.sin(heading)

        vehicle_positions[v_id] = {"x": x + x_off, "y": y + y_off,
                                   "heading": heading, "speed": speed}

        loc_msg = f"LOC_UPDATE:obj{v_id},{x + x_off},{y + y_off},{z},{angle},{v_x},{v_y},0"
        if v_id not in active_vehicles:
            print(f"  + New vehicle: {v_id}")
            add_new_car(v_id)
            active_vehicles.add(v_id)
        rt.manage_location_message(loc_msg, struc)

    # ── Render once per timestep, after all vehicles are positioned ───────────
    scene.render_to_file(
        filename=f"/home/rpegurri/Tokyo JSAC/van3twin-py/movie/frame_{timestamp}.png",
        camera="preview",
        resolution=(720, 480),
        num_samples=128,
        show_orientations=False
    )
    print(f"  Done. Exported frame for {timestamp}")

## 7 · Graph generator

In [ ]:
import topology.graph_generator as gg
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Configuration
CSV_PATH = "/home/rpegurri/Tokyo JSAC/van3twin-py/simulation_dataset_1_SUMO_162.6s-343.0s.csv"
RSU_ID = "rsu_3"
MAX_HOPS = 2
BBOX = (177.79, 239.38, 177.73, 213.38)

# Define filter for valid edges
filter = gg.create_composite_filter(
    gg.create_rssi_filter(-65.0),
    gg.create_throughput_filter(5e3)
)

In [ ]:
# Create links_snapshot element from the csv timestamp 320.0
timestamp = 336.0
df = pd.read_csv(CSV_PATH)
links_snapshot = df[df["timestamp"] == timestamp]
links_snapshot = links_snapshot.to_dict(orient="records")

### Generate

In [ ]:
# Generate graph
uplink, downlink, metadata = gg.generate_graphs(
    links_snapshot=links_snapshot,
    timestamp=342.9,
    rsu_id=RSU_ID,
    bbox=BBOX,
    max_hops=MAX_HOPS,
    link_reachability_fn=filter
)

### Visualize

In [ ]:
# Visualize uplink graph with RSSI-based edge coloring
fig, ax, pos = gg.visualize_graph(
    downlink,
    title=f"Graph for: {RSU_ID}",
    figsize=(14, 10),
    rsu_color="red",
    car_color="lightblue",
    edge_color_attr="throughput_kbps",  # Color edges by RSSI
    show_labels=True,
    layout="circular",
)
plt.show()

# Utilities

### a. CSV dataset merger

In [ ]:
import pandas as pd
from pathlib import Path

# Define the directory and input files
data_dir = Path(".")
input_files = [
    "simulation_dataset_1_SUMO_0s-162.6s.csv",
    "simulation_dataset_1_SUMO_162.6s-343.0s.csv",
    "simulation_dataset_1_SUMO_343.0s-522.8s.csv",
    "simulation_dataset_1_SUMO_522.8s-612.9s.csv",
]

# Load all CSV files
print("Loading CSV files...")
dataframes = []
for file in input_files:
    file_path = data_dir / file
    df = pd.read_csv(file_path)
    print(f"  Loaded {file}: {len(df)} rows")
    dataframes.append(df)

# Concatenate all dataframes
print(f"\nConcatenating {len(dataframes)} files...")
merged_df = pd.concat(dataframes, ignore_index=True)
print(f"  Total rows before deduplication: {len(merged_df)}")

# Remove complete row duplicates (keep first occurrence)
print("\nRemoving complete row duplicates...")
merged_df = merged_df.drop_duplicates(keep='first')
print(f"  Total rows after deduplication: {len(merged_df)}")

# Sort by timestamp
print("\nSorting by timestamp...")
merged_df = merged_df.sort_values('timestamp').reset_index(drop=True)

# Save merged file
output_file = data_dir / "simulation_dataset_merged.csv"
merged_df.to_csv(output_file, index=False)
print(f"\n✓ Merged file saved to: {output_file}")
print(f"  Total rows: {len(merged_df)}")
print(f"  Columns: {list(merged_df.columns)}")
print(f"  Timestamp range: {merged_df['timestamp'].min():.1f}s - {merged_df['timestamp'].max():.1f}s")